In [1]:
import json
import requests

sample_json = '''
{
  "station": "USC00305800",
  "name": "New York Central Park",
  "location": {
    "latitude": 40.7789,
    "longitude": -73.9692
  },
  "observations": [
    {"date": "2023-01-01", "temperature": 32, "precipitation": 0.0},
    {"date": "2023-01-02", "temperature": 28, "precipitation": 0.5},
    {"date": "2023-01-03", "temperature": 35, "precipitation": 0.0},
    {"date": "2023-01-04", "temperature": 38, "precipitation": 0.2},
    {"date": "2023-01-05", "temperature": 41, "precipitation": 0.0}
  ]
}
'''

In [2]:
data = json.loads(sample_json)

print("Station:", data['station'])
print("Location:", data['location'])
print("First observation:", data['observations'][0])

Station: USC00305800
Location: {'latitude': 40.7789, 'longitude': -73.9692}
First observation: {'date': '2023-01-01', 'temperature': 32, 'precipitation': 0.0}


In [3]:
print("Date, Temperature")
for obs in data['observations']:
    print(obs["date"], ",", obs["temperature"])

Date, Temperature
2023-01-01 , 32
2023-01-02 , 28
2023-01-03 , 35
2023-01-04 , 38
2023-01-05 , 41


In [4]:
total_temp = 0
count = 0

for obs in data["observations"]:
    total_temp += obs["temperature"]
    count += 1

avg_temp = total_temp / count
print(f"Average temperature: {avg_temp:.2f}°F")

Average temperature: 34.80°F


In [11]:
print("\nDays with precipitation:")
for obs in data["observations"]:
    if obs["precipitation"] > 0:
        print(obs["date"], "-", obs["precipitation"])


Days with precipitation:
2023-01-02 - 0.5
2023-01-04 - 0.2


In [12]:
import requests

# Step A: Get the forecast endpoint for a lat/lon
points_url = "https://api.weather.gov/points/40.78,-73.97"
r = requests.get(points_url, headers={"User-Agent": "Mozilla/5.0"})
points = r.json()

forecast_url = points["properties"]["forecast"]

# Step B: Get the forecast JSON
r2 = requests.get(forecast_url, headers={"User-Agent": "Mozilla/5.0"})
forecast = r2.json()

# Print first few forecast periods
periods = forecast["properties"]["periods"][:5]
for p in periods:
    print(p["name"], p["temperature"], p["temperatureUnit"], "-", p["shortForecast"])



Tonight 35 F - Mostly Cloudy then Slight Chance Light Rain
Tuesday 46 F - Slight Chance Light Rain then Partly Sunny
Tuesday Night 38 F - Mostly Cloudy
Wednesday 41 F - Light Rain Likely
Wednesday Night 39 F - Light Rain Likely


In [7]:
%pip install pooch requests


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [8]:
import pooch
import os

/opt/sw/anaconda3/2023.09/envs/pangeo23/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pooch
import os
print("pooch version:", pooch.__version__)
print("pooch file:", pooch.__file__)

pooch version: v1.9.0
pooch file: /home/fc2831/.local/lib/python3.11/site-packages/pooch/__init__.py


In [3]:
import pooch
import os

file_path = pooch.retrieve(
    url="https://github.com/pandas-dev/pandas/raw/main/doc/data/air_quality_no2.csv",
    known_hash=None
)

print("File downloaded to:", file_path)
print("File exists:", os.path.exists(file_path))


SHA256 hash of downloaded file: 365ca31c9296ac200e73d357e16a3c1340f9ce6746c83bf81403046dcb374361
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.


File downloaded to: /home/fc2831/.cache/pooch/458dad453f6a48e510cd544bef1854e3-air_quality_no2.csv
File exists: True


In [4]:
import os

# 1) File size (bytes)
file_size = os.path.getsize(file_path)
print(f"File size: {file_size} bytes")

# 2) Count how many lines are in the file
line_count = 0
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        line_count += 1

print(f"Number of lines: {line_count}")

File size: 31984 bytes
Number of lines: 1036


In [9]:
my_url = "https://raw.githubusercontent.com/datasets/global-temp/master/data/monthly.csv"

my_file = pooch.retrieve(url=my_url, known_hash=None)

print("Downloaded file path:", my_file)
print("File exists:", os.path.exists(my_file))
print("File size (bytes):", os.path.getsize(my_file))


Downloaded file path: /home/fc2831/.cache/pooch/5a3e1acd6b381aee143d16b9a8cd3e5e-monthly.csv
File exists: True
File size (bytes): 83924


In [11]:
with open(my_file, "r", encoding="utf-8") as f:
    for i in range(5):
        print(f.readline().strip())

Source,Year,Mean
gcag,1850-01,-0.6746
gcag,1850-02,-0.3334
gcag,1850-03,-0.5913
gcag,1850-04,-0.5887


In [13]:
import requests

base_url = "http://iridl.ldeo.columbia.edu/expert/SOURCES/.NOAA/.NCEP/.CPC/.UNIFIED_PRCP/.GAUGE_BASED/.GLOBAL/.v1p0/.Monthly/.RETRO/.rain/dods"

dds_url = base_url + ".dds"
response = requests.get(dds_url)

print("Dataset Structure (first 500 chars):")
print(response.text[:500])

Dataset Structure (first 500 chars):
Dataset {
    Float32 T[T = 324];
    Float32 Y[Y = 360];
    Float32 X[X = 720];
    Grid {
     ARRAY:
        Float32 rain[T = 324][Y = 360][X = 720];
     MAPS:
        Float32 T[T = 324];
        Float32 Y[Y = 360];
        Float32 X[X = 720];
    } rain;
} rain;



In [14]:
das_url = base_url + ".das"
response = requests.get(das_url)

print("Dataset Attributes (first 1000 chars):")
print(response.text[:1000])


Dataset Attributes (first 1000 chars):
Attributes {
    X {
        String standard_name "longitude";
        Float32 pointwidth 0.5;
        Int32 gridtype 1;
        String units "degree_east";
    }
    T {
        Float32 pointwidth 1.0;
        String calendar "360";
        Int32 gridtype 0;
        String units "months since 1960-01-01";
    }
    Y {
        String standard_name "latitude";
        Float32 pointwidth 0.5;
        Int32 gridtype 0;
        String units "degree_north";
    }
    rain {
        Int32 pointwidth 0;
        String standard_name "lwe_precipitation_rate";
        Float32 file_missing_value -999.0;
        String history "Boxes with less than 0.0% dropped";
        Float32 missing_value NaN;
        String units "mm/day";
        String long_name "Monthly Precipitation";
    }
NC_GLOBAL {
    String Conventions "IRIDL";
}
}



## 1. Identify dimensions and variables

- **What are the dimension names?**  
  T (time) = 324, Y (latitude) = 360, X (longitude) = 720.

- **What is the main variable?**  
  The main variable is **rain**.

## 3. Document what you learned

- **What does this dataset contain?**  
  Global monthly precipitation data (rainfall).

- **What time period does it cover?**  
  Time is measured in **months since 1960-01-01**. With 324 time steps, it covers approximately **1960–1987**.

- **What geographic region does it cover?**  
  Global coverage (360 latitude points × 720 longitude points).

- **What are the units of the main variable?**  
  **mm/day** (millimeters per day).